# Document Ingestion

## Trying different document ingestion libraries

### 1. LangChain

In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="this is the text content of the document for RAG testing.",
    metadata = {                        # helps applying filters based on document (Qdrant)
        "source": "example.txt",
        "pages": 1,
        "author": "Nirav",
        "date_created": "2025-07-11",
    }
)

print(doc)

page_content='this is the text content of the document for RAG testing.' metadata={'source': 'example.txt', 'pages': 1, 'author': 'Nirav', 'date_created': '2025-07-11'}


In [3]:
## create a simple txt file
import os

os.makedirs("../data/text_files", exist_ok=True)
os.makedirs("../data/pdf_files", exist_ok=True)

In [4]:
sample_text = """
LangChain is a framework for developing applications powered by language models. It can be used for chatbots, Generative Question-Answering (GQA), summarization, and much more.
"""

with open("../data/text_files/sample_01.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

In [5]:
### Textloader
from langchain_community.document_loaders import TextLoader

text_loader = TextLoader("../data/text_files/sample_01.txt", encoding="utf-8")
document = text_loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/sample_01.txt'}, page_content='\nLangChain is a framework for developing applications powered by language models. It can be used for chatbots, Generative Question-Answering (GQA), summarization, and much more.\n')]


In [6]:
sample_text_2 = """
    Python is a high-level, interpreted programming language known for its readability and versatility. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python is widely used in web development, data analysis, artificial intelligence, scientific computing, and more.
"""

with open("../data/text_files/sample_02.txt", "w", encoding="utf-8") as f:
    f.write(sample_text_2)

In [7]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls = TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
)

documents = dir_loader.load()
print(documents)

100%|██████████| 2/2 [00:00<00:00, 999.83it/s]

[Document(metadata={'source': '..\\data\\text_files\\sample_01.txt'}, page_content='\nLangChain is a framework for developing applications powered by language models. It can be used for chatbots, Generative Question-Answering (GQA), summarization, and much more.\n'), Document(metadata={'source': '..\\data\\text_files\\sample_02.txt'}, page_content='\n    Python is a high-level, interpreted programming language known for its readability and versatility. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python is widely used in web development, data analysis, artificial intelligence, scientific computing, and more.\n')]


### Performing PDF Document Loader

In [8]:
from langchain_docling import DoclingLoader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

SOURCE = "../data/pdf_files/Indian Institute of Management (IIM-Nagpur) - Nagpur - Internship Brochure - 2023-24.pdf"
output_dir = "../data/scratch"
filename = SOURCE.split("/")[-1].replace(".pdf", ".md")

pymupdf_loader = PyMuPDFLoader(SOURCE)

pdf_document = pymupdf_loader.load()


c:\Users\Nirav\Documents\My Learning\agentic-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import pprint

print(pdf_document)
pprint.pp(pdf_document[0].metadata)

output_path = os.path.join(output_dir, filename)
full_content = "\n\n".join([doc.page_content for doc in pdf_document])

pprint.pp(full_content[:1000])

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Windows)', 'creationdate': '2023-08-28T19:14:26+05:30', 'source': '../data/pdf_files/Indian Institute of Management (IIM-Nagpur) - Nagpur - Internship Brochure - 2023-24.pdf', 'file_path': '../data/pdf_files/Indian Institute of Management (IIM-Nagpur) - Nagpur - Internship Brochure - 2023-24.pdf', 'total_pages': 78, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-10-25T15:46:40+05:30', 'trapped': '', 'modDate': "D:20231025154640+05'30'", 'creationDate': "D:20230828191426+05'30'", 'page': 0}, page_content=''), Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Windows)', 'creationdate': '2023-08-28T19:14:26+05:30', 'source': '../data/pdf_files/Indian Institute of Management (IIM-Nagpur) - Nagpur - Internship Brochure - 2023-24.pdf', 'file_path': '../data/pdf_files/Indian Institute of Management (IIM-Nagpur) - Na

from docling.document_converter import DocumentConverter


converter = DocumentConverter()
result = converter.convert(SOURCE)

print(result.document.export_to_markdown)

In [10]:
os.makedirs("../data/scratch", exist_ok=True)

output_dir = "../data/scratch"
filename = SOURCE.split("/")[-1].replace(".pdf", ".md")

print(result)
result.document.save_as_markdown(os.path.join(output_dir, filename))

### Building a pipeline to go through direcctory and ingest all documents

In [73]:
from langchain_community.document_loaders import PyMuPDFLoader
import os
from pathlib import Path

def ingest_documents(file_path: str):
    pymupdf_loader = PyMuPDFLoader(file_path)
    pdf_document = pymupdf_loader.load()
    return pdf_document
    
SOURCE = "../data/pdf_files/financial_reports/"
output_dir = "../data/scratch/"
os.makedirs(output_dir, exist_ok=True)

# documents = []
# pdf_files = list(Path(SOURCE).rglob("*.pdf"))
# for file in os.listdir(SOURCE):
#     if file.endswith(".pdf"):
#         file_path = os.path.join(SOURCE, file)
#         pdf_document = ingest_documents(file_path)
#         documents.append(pdf_document)

dir_loader = DirectoryLoader(
    SOURCE,
    glob="**/*.pdf",
    loader_cls = PyMuPDFLoader,
    show_progress=False,
)

documents = dir_loader.load()

In [74]:
import pprint

# Store documents in scratch directory
pprint.pp(documents)

[Document(metadata={'producer': 'Adobe Acrobat (32-bit) 25.1.20474', 'creator': 'Adobe Acrobat (32-bit) 25.1.20474', 'creationdate': '2025-05-24T00:27:11+05:30', 'source': '..\\data\\pdf_files\\financial_reports\\18f91098-8691-409d-b771-808c48964a4f.pdf', 'file_path': '..\\data\\pdf_files\\financial_reports\\18f91098-8691-409d-b771-808c48964a4f.pdf', 'total_pages': 591, 'format': 'PDF 1.6', 'title': '', 'author': 'Wadia Ghandy & Co.', 'subject': '', 'keywords': '', 'moddate': '2025-05-24T00:29:34+05:30', 'trapped': '', 'modDate': "D:20250524002934+05'30'", 'creationDate': "D:20250524002711+05'30'", 'page': 0}, page_content="May 24, 2025 \nSc no. - 18677 \n \nDear Sirs/Madam, \n \n \nSub: Integrated Annual Report for the Financial Year 2024-25 and Notice convening the \n80th Annual General Meeting (‘AGM’) of Tata Motors Limited (‘the Company’) \n \nFurther to our letters dated May 13, 2025 and May 21, 2025, wherein we had informed that the \n80th AGM of the Company will be held on Frida

In [75]:
## Chunking function

from typing import List, Dict, Any, Literal, Tuple
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter, SentenceTransformersTokenTextSplitter

ChunkingStrategy = Literal["recursive", "character", "sentence_transformers"]

def split_documents(
        documents: List[Any], 
        chunking_strategy: ChunkingStrategy, 
        chunk_size: int = 1000,
        chunk_overlap: int = 200,
        **kwargs: Any,
        ) -> List[Any]:
    """
    Splits documents into smaller chunks based on the specified chunking strategy.
    
    Args:
        documents (List[Any]): List of documents to be split.
        chunking_strategy (ChunkingStrategy): Strategy to use for chunking ('recursive' or 'character').
        chunk_size (int): Size of each chunk.
        chunk_overlap (int): Overlap between chunks.
        **kwargs: Additional keyword arguments for text splitters.
    
    Returns:
        List[Any]: List of chunked documents.
    """

    if chunking_strategy == "recursive":
        length_function = kwargs.get("length_function", len)
        is_separator_regex = kwargs.get("is_separator_regex", False)

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=length_function,
            is_separator_regex=is_separator_regex,
        )
        documents_chunks = []
        for document in documents:
            chunks = text_splitter.split_documents([document])
            documents_chunks.extend(chunks)
        return documents_chunks
    
    elif chunking_strategy == "character":
        model_name = kwargs.get("model_name", "gpt-4")

        text_splitter = CharacterTextSplitter(
            model_name=model_name,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

        documents_chunks = []
        for document in documents:
            chunks = text_splitter.split_documents([document])
            documents_chunks.extend(chunks)

        return documents_chunks
    elif chunking_strategy == "sentence_transformers":
        model_name = kwargs.get("model_name", "all-MiniLM-L6-v2")

        try:
            text_splitter = SentenceTransformersTokenTextSplitter(
                model_name=model_name,
                chunk_overlap=chunk_overlap,
            )

            documents_chunks = []
            for document in documents:
                chunks = text_splitter.split_documents([document])
                documents_chunks.extend(chunks)

            return documents_chunks
        except Exception as e:
            raise ValueError(f"Error initializing SentenceTransformersTokenTextSplitter: {e}")
    else:
        raise ValueError(f"Unsupported chunking strategy: {chunking_strategy}")

In [76]:
chunks = split_documents(
    documents,
    chunking_strategy="sentence_transformers",
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)
chunks

2025-11-12 16:13:52,631 - INFO - Use pytorch device_name: cpu
2025-11-12 16:13:52,633 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


[Document(metadata={'producer': 'Adobe Acrobat (32-bit) 25.1.20474', 'creator': 'Adobe Acrobat (32-bit) 25.1.20474', 'creationdate': '2025-05-24T00:27:11+05:30', 'source': '..\\data\\pdf_files\\financial_reports\\18f91098-8691-409d-b771-808c48964a4f.pdf', 'file_path': '..\\data\\pdf_files\\financial_reports\\18f91098-8691-409d-b771-808c48964a4f.pdf', 'total_pages': 591, 'format': 'PDF 1.6', 'title': '', 'author': 'Wadia Ghandy & Co.', 'subject': '', 'keywords': '', 'moddate': '2025-05-24T00:29:34+05:30', 'trapped': '', 'modDate': "D:20250524002934+05'30'", 'creationDate': "D:20250524002711+05'30'", 'page': 0}, page_content='may 24, 2025 sc no. - 18677 dear sirs / madam, sub : integrated annual report for the financial year 2024 - 25 and notice convening the 80th annual general meeting ( ‘ agm ’ ) of tata motors limited ( ‘ the company ’ ) further to our letters dated may 13, 2025 and may 21, 2025, wherein we had informed that the 80th agm of the company will be held on friday, june 20,

## Creating an Embedding and Vector DB class

In [77]:
import numpy as np
import os
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple

In [78]:
class EmbeddingModel:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding model.
        """
        self.model_name = model_name
        self.model = None
        self.model = self._load_model(model_name)

    def _load_model(self, model_name: str) -> SentenceTransformer:
        """
        Load the embedding model.
        """
        try:
            model = SentenceTransformer(model_name)
            print(f"Loaded model: {model_name} with dimension {model.get_sentence_embedding_dimension()}")
            return model
        except Exception as e:
            raise ValueError(f"Error loading model {model_name}: {e}")

    def embed_documents(self, documents: List[str]) -> List[np.ndarray]:
        """
        Embed a list of documents.
        """
        try:
            return self.model.encode(documents)
        except Exception as e:
            raise ValueError(f"Error embedding documents: {e}")

In [79]:
embedding_model = EmbeddingModel()

2025-11-12 16:14:18,958 - INFO - Use pytorch device_name: cpu
2025-11-12 16:14:18,958 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Loaded model: all-MiniLM-L6-v2 with dimension 384


In [80]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_vector_store()

    def _initialize_vector_store(self):
        """
        Initialize the Chroma vector store.
        """
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(
                path = self.persist_directory
            )
            self.collection = self.client.get_or_create_collection(name=self.collection_name)
            print(f"Initialized vector store with collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            raise ValueError(f"Error initializing vector store: {e}")
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents (List[Any]): List of documents to add.
            embeddings (np.ndarray): Corresponding embeddings for the documents.
        """
        try:
            if len(documents) != len(embeddings):
                raise ValueError("Number of documents and embeddings must match.")
            
            print(f"Adding {len(documents)} documents to vector store...")
            
            # Prepare data for insertion
            ids = []
            metadatas = []
            document_texts = []
            embeddings_list = []

            for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
                doc_id = f"doc_{str(uuid.uuid4().hex[:8])}_{i}"
                ids.append(doc_id)

                metadata = dict(doc.metadata)
                metadata['doc_index'] = i
                metadata['content_length'] = len(doc.page_content)
                metadatas.append(metadata)

                document_texts.append(doc.page_content)

                embeddings_list.append(embedding)

            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=document_texts,
                embeddings=embeddings_list
            )
            print("Documents added successfully.")
        except Exception as e:
            raise ValueError(f"Error adding documents to vector store: {e}")

In [81]:
vector_store = VectorStore(collection_name="financial_reports", persist_directory="../data/vector_store")
print(vector_store)

Initialized vector store with collection: financial_reports
Existing documents in collection: 0


In [83]:
docs = [chunk.page_content for chunk in chunks]

# generate embeddings
embeddings = embedding_model.embed_documents(docs)

# store chunks and embeddings in the vector store
batch_size = 5461

# Create batches of chunks and embeddings to avoid memory issues
for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i + batch_size]
    batch_embeddings = embeddings[i:i + batch_size]
    vector_store.add_documents(batch_chunks, batch_embeddings)

Batches: 100%|██████████| 346/346 [06:03<00:00,  1.05s/it]


Adding 5461 documents to vector store...
Documents added successfully.
Adding 5461 documents to vector store...
Documents added successfully.
Adding 141 documents to vector store...
Documents added successfully.


## Retrieval Pipeline

In [ ]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_model: EmbeddingModel, top_k: int = 5):
        """
        Initialize the RAG Retriever.
        """
        self.vector_store = vector_store
        self.embedding_model = embedding_model
        self.top_k = top_k

    def retrieve(self, query: str, top_k: int = None, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for the given query.

        Args:
            query (str): The input query string.
            score_threshold (float): Minimum similarity score to consider a document relevant.
        Returns:
            List[Dict[str, Any]]: List of retrieved documents with metadata and similarity scores.
        """
        top_k = top_k if top_k is not None else self.top_k
        try:
            query_embedding = self.embedding_model.embed_documents([query])[0]
            results = self.vector_store.collection.query(
                query_embeddings=query_embedding,
                n_results=top_k,
            )

            retrieved_docs = []
            doc_count = 0

            if results and 'documents' in results and len(results['documents']) > 0:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc, metadata, distance, doc_id) in enumerate(zip(documents, metadatas, distances, ids)):
                    similarity_score = 1 / (1 + distance)  # Convert distance to similarity score

                    if similarity_score >= score_threshold:
                        doc_count += 1
                        retrieved_docs.append({
                            "document": doc,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "id": doc_id
                        })

                print(f"Retrieved {doc_count} documents above the score threshold of {score_threshold}.")

            return retrieved_docs

        except Exception as e:
            print(f"Error retrieving documents: {e}")
            return []

    

In [84]:
query = "Waht was the capex for Bajaj Finance in the last quarter?"

retriever = RAGRetriever(vector_store, embedding_model, top_k=5)
relevant_docs = retriever.retrieve(query, score_threshold=0.5)
print("Relevant Documents:")
# for i, doc in enumerate(relevant_docs):
pprint.pp(f"\n{[doc['similarity_score'] for doc in relevant_docs]}\n")

Batches: 100%|██████████| 1/1 [00:00<00:00, 86.85it/s]

Retrieved 5 documents above the score threshold of 0.5.
Relevant Documents:
('\n'
 '[0.5375471376984705, 0.5129643127950493, 0.5078055214339176, '
 '0.5058857879162426, 0.5029136361191958]\n')


## Integrating LLM with retrieval pipeline

In [ ]:
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in environment variables.")

# 1. Initialize ChatGroq LLM
chat_groq = ChatOllama(
                model="llama3.2:latest",
                temperature=0.2,
                num_predict=1000
            )

# 2. RAG function

def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['document'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    prompt=f"""Use the following context to answer the question concisely with precision.
            Context: 
            {context}

            Question: 
            {query}

            Answer:"""
    
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

### Enhanced RAG pipeling

In [91]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - minimum similarity score filtering
    - returns confidence score
    - option to return context/source along with answer
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {
                'answer': 'No relevant context found.',
                'sources': [],
                'confidence_score': 0.0,
                'context': ''
            }
    
    context = "\n\n".join([doc['document'] for doc in results])
    sources =[{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['document'][:200] + '...'
    } for doc in results]

    confidence = np.max([doc['similarity_score'] for doc in results])

    prompt = f"""Use the following context to answer the question concisely with precision. If the answer is not found in the context, respond with 'Information not available in the provided context.'.
            Context: 
            {context}

            Question: 
            {query}

            Answer:"""
    
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence_score': confidence,
        'context': context if return_context else None
    }

    return output

In [92]:
query = "What is the net total income in H1 of FY2026 for Bajaj Finance?"

simple_response = rag_simple(query, retriever, chat_groq, top_k=10)
print("Simple RAG Response:")
pprint.pp(simple_response)

Batches: 100%|██████████| 1/1 [00:00<00:00, 82.96it/s]


Retrieved 10 documents above the score threshold of 0.0.


2025-11-12 16:44:09,190 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Simple RAG Response:
'The net total income for H1 of FY2026 for Bajaj Finance was ₹ 25,780 crore.'


In [93]:
enhanced_response = rag_advanced(
    query, retriever, chat_groq, 
    top_k=10, min_score=0.5, 
    return_context=True)

print("\nEnhanced RAG Response:")
pprint.pp(enhanced_response)

Batches: 100%|██████████| 1/1 [00:00<00:00, 65.46it/s]

Retrieved 10 documents above the score threshold of 0.5.



2025-11-12 16:44:10,945 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"



Enhanced RAG Response:
{'answer': 'Net total income for H1 FY26 increased by 21% to ₹ 25,780 crore '
           'from ₹ 21,365 crore in H1 FY25.',
 'sources': [{'source': '..\\data\\pdf_files\\financial_reports\\BJAF22330_Half-Yearly-Communication.pdf',
              'page': 2,
              'score': 0.6184514458277496,
              'preview': 'september 2025. net interest income for h1 fy26 '
                         'increased by 22 % to ₹ 21, 012 crore from ₹ 17, 203 '
                         'crore in h1 fy25. net total income for h1 fy26 '
                         'increased by 21 % to ₹ 25, 780 crore from ₹ 21, 365 '
                         'c...'},
             {'source': '..\\data\\pdf_files\\financial_reports\\BJAF22330_Half-Yearly-Communication.pdf',
              'page': 2,
              'score': 0.6035142154779528,
              'preview': 'from ₹ 21, 365 crore in h1 fy25. operating expenses '
                         'to net total income for h1 fy26 was 32. 7 % as '
  